In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor

x , y = load_iris(as_frame=True , return_X_y = True)
x_train , x_test , y_train , y_test = train_test_split(x , y ,test_size=0.2)

dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(x_train , y_train)
dummy_regr.predict(x_test)
dummy_regr.score(x_test,y_test)

-0.025066844919785725

### How is Linear Regression Model Trained?

Linear Regression model ko train karne ke liye 2 main approaches hoti hain:

- **Normal Equation (Direct Mathematical Solution)**
- **Iterative Optimization (Stochastic Gradient Descent - SGD)**

---

#### Step 1: Instantiate the Model

**Option 1: Normal Equation (LinearRegression)**

```python
from sklearn.linear_model import LinearRegression
linear_regressor = LinearRegression()
```

📌 This internally uses **direct mathematical solvers (like SVD / Normal Equation concept)**, not Gradient Descent.

---

**Option 2: Iterative Optimization (SGDRegressor)**

```python
from sklearn.linear_model import SGDRegressor
linear_regressor = SGDRegressor()
```

📌 This uses **Stochastic Gradient Descent (SGD)** for optimization.

---

#### Step 2: Fit the Model

```python
# X_train → feature matrix
# y_train → target vector

linear_regressor.fit(X_train, y_train)
```

👉 This step trains the model and learns parameters (weights).

---

### Mathematical Intuition

#### 1. Hypothesis Function

Linear regression predicts output as:

$$
\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n
$$

Matrix form:

$$
\hat{y} = X\theta
$$

---

#### 2. Cost Function (Mean Squared Error)

Model tries to minimize error:

$$
J(\theta) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2
$$

---

#### 3. Normal Equation (Direct Solution)

Best parameters directly compute hote hain:

$$
\theta = (X^T X)^{-1} X^T y
$$

📌 Characteristics:
- One-step solution
- Exact answer (mathematically)
- Slow for large datasets (matrix inverse expensive)

---

#### 4. Stochastic Gradient Descent (SGD)

Instead of direct solution, parameters iteratively update hote hain:

$$
\theta_j := \theta_j - \alpha \frac{\partial}{\partial \theta_j} J(\theta; x^{(i)}, y^{(i)})
$$

📌 Meaning:
- $\alpha$ = learning rate
- Update happens using **one sample at a time**
- Faster for large datasets
- Slightly noisy but efficient

---

### Final Intuition

- **LinearRegression**
  → Direct mathematical solution (Normal Equation / SVD based)

- **SGDRegressor**
  → Iterative learning using **Stochastic Gradient Descent**

👉 Goal same hai:
> Best-fit line find karna jisme prediction error minimum ho

## SGDRegressor

### SGDRegressor (Stochastic Gradient Descent) Ekdam Depth Mein

#### 1. Implements Stochastic Gradient Descent (SGD)
*   **Basic (Layman):** Maan lo tum aankh band karke kisi pahad (mountain) ki choti se niche utar rahe ho. Normal Gradient Descent me tum poore pahad ka naksha ek sath dekhte ho (saara data) aur phir ek kadam badhate ho. **Stochastic (random)** me tum sirf apne aas-paas ka ek pathar dekhte ho (ek data point ya mini-batch) aur turant ek kadam niche rakh dete ho. Yeh thoda zig-zag hota hai, par bahut fast hota hai.
*   **Advance (Maths):** Machine Learning me hum loss function $J(\theta)$ ko minimize karte hain. Normal OLS (Ordinary Least Squares) me parameter update $\theta = \theta - \eta \nabla J(\theta)$ poore dataset $N$ par calculate hota hai. SGD me, gradient sirf ek random sample $i$ par calculate hota hai: 
    $$ \theta_{t+1} = \theta_t - \eta \nabla J_i(\theta_t) $$
    Isse computation ek step ke liye $\mathcal{O}(N)$ se ghat kar $\mathcal{O}(1)$ ho jati hai.

#### 2. Use for large training set up (> 10k samples)
*   **Basic:** Agar data chhota hai (jaise 1,000 rows), toh purana `LinearRegression` best hai. Par jab data me lakhon rows hon, toh purana tarika computer ko hang kar dega (Out of Memory). Wahan SGD king hai.
*   **Advance:** Normal Linear Regression ek exact solution nikalta hai Normal Equation se: $\theta = (X^T X)^{-1} X^T y$. Isme matrix inversion ki time complexity $\mathcal{O}(f^3)$ hoti hai (jahan $f$ features hain). Jab dataset $> 10,000$ aur features bahut jyada hon, toh yeh equation RAM ko saturate kar deti hai. SGD memory-efficient hai kyunki yeh data ko chunks (batches) me process karta hai (Out-of-core learning).

#### 3. Hyperparameters for Loss Function (`loss`)
Loss batata hai ki prediction $\hat{y}$ aur actual $y$ me kitna gap hai. SGD is galti ko measure karne ke tarike me variations deta hai.

*   **`loss = 'squared_error'`**
    *   **Basic:** Jo galti hui, uska square kar do. (2 ki galti = 4 ka loss, 10 ki galti = 100 ka loss). 
    *   **Advance:** Yeh standard Mean Squared Error (MSE) hai.
        $$ L = \frac{1}{2} (y - \hat{y})^2 $$
        **Use Case:** Yeh convex (bowl-shape) hota hai aur mathematically solve karna easy hai. Lekin agar data me **Outliers** (extreme values) hain, toh square karne se error bahut bada ho jata hai aur gradient ekdam bhatak jata hai.
*   **`loss = 'huber'`**
    *   **Basic:** Yeh thoda smart hai. Chhoti galtiyon ke liye yeh `squared_error` ki tarah behave karta hai, par agar galti bahut badi (outlier) ho, toh yeh square karna band karke use normal linear galti maan leta hai.
    *   **Advance:** Yeh mathematically ek piecewise function hai jise ek threshold $\epsilon$ (epsilon) control karta hai.
        *   Agar $|y - \hat{y}| \le \epsilon$, toh loss = $\frac{1}{2}(y - \hat{y})^2$
        *   Agar $|y - \hat{y}| > \epsilon$, toh loss = $\epsilon |y - \hat{y}| - \frac{1}{2}\epsilon^2$
        **Use Case:** Jab tumhare dataset me noise ya bahut outliers hon aur tum nahi chahte ki unki wajah se model ki line kharab ho.

#### 4. Hyperparameters for Penalty (`penalty`)
Penalty (Regularization) model ko "over-smart" (overfitting) hone se rokti hai. Yeh model ke weights ($w$) ko chhota rakhne ke liye loss me ek extra tax jodti hai.

*   **`penalty = 'l2'` (Ridge Regularization)**
    *   **Basic:** Saare features ke weights ko daba kar chhota karta hai, par kisi ko exactly 0 nahi karta.
    *   **Advance:** Loss function me weights ka squared sum add hota hai.
        $$ J = Loss + \alpha \sum_{i=1}^{n} w_i^2 $$
        (Yahan $\alpha$ penalty ki takat hai). Gradient nikalte waqt yeh weight update me ek decay term ($-2\alpha w_i$) jod deta hai.
*   **`penalty = 'l1'` (Lasso Regularization)**
    *   **Basic:** Yeh strict teacher ki tarah hai. Jo features faltu hain, unka weight exactly $0$ kar deta hai. Isliye yeh **Feature Selection** ka kaam bhi karta hai.
    *   **Advance:** Loss me weights ka absolute sum add hota hai.
        $$ J = Loss + \alpha \sum_{i=1}^{n} |w_i| $$
        Iska mathematical constraint space ek diamond shape (L1 norm ball) banata hai. Optimization aksar is diamond ke corners par hit karta hai, jahan kuch axes $0$ hoti hain, leading to sparse solutions (many zero weights).
*   **`penalty = 'elasticnet'`**
    *   **Basic:** L1 aur L2 dono ka combination. L1 features ko zero karta hai, aur L2 bache hue features ko stable rakhta hai.
    *   **Advance:** 
        $$ J = Loss + \alpha \cdot \rho \sum |w_i| + \alpha (1-\rho) \frac{1}{2} \sum w_i^2 $$
        (Jahan $\rho$ `l1_ratio` hai). L1 akele tab fail ho jata hai jab features highly correlated hon (wo randomly ek ko chun lega aur baaki ko hata dega). ElasticNet L2 ki madad se saare correlated features ko proportionally distribute karta hai.

#### 5. Hyperparameters for Learning Rate (`learning_rate`)
Learning Rate ($\eta$) decide karta hai ki weights update karte waqt model kitna bada "kadam" (step size) lega. 

*   **`learning_rate = 'constant'`**
    *   **Basic:** Step size hamesha ek jaisa rahega ($\eta = \text{eta0}$).
    *   **Advance:** $\eta_t = \eta_0$. Danger yeh hai ki agar learning rate thoda sa bhi bada hua, toh global minima par model rukh nahi payega aur bounce karta rahega (divergence).
*   **`learning_rate = 'optimal'`**
    *   **Basic:** Jaise-jaise training aage badhegi, model apne kadam chhote karta jayega taaki wo exact answer par aakar aaram se ruk sake.
    *   **Advance:** Yeh Leon Bottou ka heuristic formula use karta hai:
        $$ \eta_t = \frac{1}{\alpha (t_0 + t)} $$
        ($t$ time step hai). Yeh theoretically guarantee deta hai ki model converge karega, par real world me ise tune karna thoda tricky hota hai.
*   **`learning_rate = 'invscaling'`**
    *   **Basic:** Yeh bhi time ke sath step size chhota karta hai par ek alag power use karke.
    *   **Advance:** 
        $$ \eta_t = \frac{\eta_0}{t^{power\_t}} $$
        (Default `power_t = 0.25`). Yeh start me bade steps leta hai aur baad me smooth convergence deta hai.
*   **`learning_rate = 'adaptive'`**
    *   **Basic:** Yeh sabse smart hai. Model constant rate se shuru karega. Par jab usko lagega ki "loss ab kam nahi ho raha, main atak gaya hu", toh wo turant apne step size ko 5 guna chhota kar lega.
    *   **Advance:** Starts with $\eta_0$. If training loss fails to decrease by at least `tol` for `n_iter_no_change` consecutive epochs, the learning rate is divided by 5: $\eta_{new} = \frac{\eta}{5}$. Yeh complex real-world data ke liye highly effective hai.

#### 6. Early Stopping (`early_stopping`)
*   **`early_stopping = 'True'`**
    *   **Basic:** Model ek internal test set banata hai. Agar model ko lagta hai ki usko aur sikhane se koi fayda nahi ho raha (ya wo overfit karne laga hai), toh wo samay bachane ke liye training beech me hi rok deta hai.
    *   **Advance:** A fraction of training data (`validation_fraction`) is set aside as a validation set. If the validation score doesn't improve by at least `tol` for `n_iter_no_change` epochs, training stops. Yeh generalization error ko minimize karta hai.
*   **`early_stopping = 'False'`**
    *   **Basic/Advance:** Model strictly apne pure `max_iter` (epochs) complete karega, chahe loss reduce hona bahut pehle hi kyu na band ho gaya ho.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

x , y = fetch_california_housing(as_frame=True , return_X_y=True)
x_train , x_test , y_train , y_test = train_test_split(x,y , test_size=0.2 , random_state=42)

sgd = Pipeline([
    ("std_scaler" , StandardScaler()),
    ("regressor" , SGDRegressor(shuffle=True , max_iter=100))
])

sgd_trained = sgd.fit(x_train,y_train)

preds = sgd_trained.predict(x_test)

mean_squared_error(y_test,preds)


0.5515214688282113

### Normal Equaltion :- 
```python
from sklearn.linear_model import LinearRegression
lin_reg=LinearReression()
```
### Iterative Equation :-
```python
from sklearn.linear_model import SGDRegressor
sgd_reg=SGDRegressor()
```

### SGDRegressor :-
* loss => 
    * (a) 'suqares error' 
    * (b) 'huber'
* penalty => 
    * (a) 'l1'  
    * (b) 'l2'  
    * (c) 'elasticnet'
* learning_rate => 
    * (a) 'constant' 
    * (b) 'optimal' 
    * (c) 'invscaling'  
    * (d)  'adaptive'
* early_stopping => 
    * (a) 'True' 
    * (b) 'False'

In [8]:
from sklearn.linear_model import SGDRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline


x , y  = fetch_california_housing(as_frame=True , return_X_y = True)
x_train , x_test , y_train , y_test = train_test_split(x,y , test_size=0.2 , random_state=42)

sgd_reg = SGDRegressor(random_state = 42)

sgd = Pipeline([
    ("std_scaler" , StandardScaler()),
    ("regressor" , sgd_reg)
])

sgd_model = sgd.fit(x_train , y_train)
preds = sgd_model.predict(x_test)

mse_error = mean_squared_error(y_test , preds)
mse_error

0.5505987775857771

> ####  `shuffle=True` : Data will get shuffle after each epoch

``` python
sgd_reg = SGDRegressor(shuffle=True)
```


### Learning Rate (eta0) in SGDRegressor

**Learning Rate** (jise hum code mein $\eta_0$ ya `eta0` kehte hain) ka simple matlab hai: Aapke kadam ki lambaai ya gaadi ki speed. Aapko ek exact point (Minimum Loss) par gaadi park karni hai. Ab alag-alag learning rates gaadi chalane ke alag-alag styles hain.

Scikit-Learn (`SGDRegressor`) mein mainly 4 tarah ke learning rate hote hain:

#### 1. constant (Ek hi speed se chalna)
* **Kya hai:** Aapne shuru se lekar end tak apne kadam ki lambaai ek dam fix rakhi hai (e.g., `eta0=0.01`).
* **Math:** Kisi bhi step $t$ par aapki speed hamesha fixed rahegi:
  $$\eta^{(t)} = \eta_0$$ 
* **Analogy:** Jaise highway par gaadi ko 'Cruise Control' par daal dena. Speed na kam hogi, na zyada.
* **Kyun aur Kab use karein:** Ise tabhi use karein jab aap `average=True` (Averaged SGD) use kar rahe hon. Kyunki ASGD mein average nikalne se galtiyan chup jati hain, toh hum constant high speed rakh sakte hain.

#### 2. invscaling (Inverse Scaling - Dheere-dheere speed kam karna)
* **Kya hai:** Jaise-jaise training (epochs $t$) aage badhti hai, aapka learning rate lagatar chhota hota jata hai.
* **Math:** 
  $$\eta^{(t)} = \frac{\eta_0}{t^p}$$ 
  *(Jahan $t$ current step hai, aur $p$ yaani `power_t` ek parameter hai jo speed kam hone ka rate tay karta hai).*
* **Analogy:** Jaise-jaise aap apni destination ke paas pahunchte hain, aap automatically gaadi ki speed dheere karne lagte hain.
* **Kyun aur Kab use karein:** Yeh `SGDRegressor` ka **Default** tarika hai. Ordinary data par jab aap normal SGD chala rahe hon toh ise default rehne dein. Yeh safe hota hai.

#### 3. adaptive (Smart Speed - Jab zarurat ho tabhi brake lagao)
* **Kya hai:** Model shuru mein fix speed se chalta rehta hai. Par jaise hi lagatar 5 epochs tak koi fayda (improvement) nahi hota, toh wo apni speed kam (divide by 5) kar deta hai.
* **Math:** Agar model ko lagta hai ki loss improve nahi ho raha, toh nayi speed aadhi ho jayegi:
  $$\eta^{(t)} = \frac{\eta^{(t-1)}}{5}$$
* **Analogy:** Naye shehar mein tez speed mein gaadi chala rahe ho. Jaise hi lagta hai bhatak gaye ho, turant speed aadhi kar dete ho taaki aaram se rasta dhundh sako.
* **Kyun aur Kab use karein:** Jab aapka data bohot complex ho aur aapko samajh na aa raha ho ki kaunsa learning rate best hoga. Yeh model ko bina bhatke center tak pahunchne mein madad karta hai.

#### 4. optimal (Leon Bottou's formula)
* **Kya hai:** Yeh ek complex math formula par aadharit hai jo data ki situation dekh kar khud tay karta hai ki shuruwati aur baad ki speed kya honi chahiye. Isme aapko `eta0` dene ki zarurat hi nahi padti.
* **Math:** 
  $$\eta^{(t)} = \frac{1}{\alpha (t_0 + t)}$$ 
  *(Jahan $\alpha$ regularization parameter hai, aur $t_0$ ek fix base time step hai).*
* **Kyun aur Kab use karein:** Jab aapke paas hyperparameter tuning (trial and error) ka time na ho aur aap chahte ho ki model khud ek achhi speed pakad le.



> #### `1. learning_rate='invscaling'` : By default
eta0 = 1e-2 , power_t = 0.25 
``` python
from sklearn.linear_model import SGDRegressor
linear_regressor = SGDRegressor(
    learning_rate="invscaling",
    random_state=42)

```
> #### `2. learning_rate='constant'`
```python
from sklearn.linear_model import SGDRegressor
linear_egressor = SGDRegressor(
    learning_rate="constant",
    eta0=1e-2,
    random_state=42)
```

> #### `3. learning_rate='adaptive'`
```python
from sklearn.linear_model import SGDRegressor
linear regressor = SGDRegressor(
    learning_rate="adaptive",
    eta0=1e-2,
    random_state=42)

### Stopping Criteria

> #### Option 1: tolarance(tol) , max_iter(maximum epoch) , n_iter_no_change(if convergance stops run more n times then stop)

#### Parameters Definition

* **max_iter**: Maximum number of epochs. (Model poore dataset ko zyada se zyada kitni baar process karega).
* **tol (Tolerance)**: Improvement ki limit. Yeh wo minimum loss reduction hai jo hum har epoch mein expect karte hain (e.g., $10^{-3}$ ya $0.001$).
* **n_iter_no_change**: Patience level. Agar lagatar itne epochs tak model ka improvement $tol$ ki value se chhota reh jata hai, toh training wahin stop ho jayegi.
\end{description}
```python
from sklearn.linear_model import SGDRegressor
linear_regressor = SGDRegressor(
    loss="squared_error",
    max_iter=100,
    tol=1e-3,
    n_iter_no_change=5
)
```

> #### Option 2: early_stopping , validation_function

#### 1. Simple Explanation (Aasan Bhasha Mein)
Pichle method mein (Option 1), model sirf apna training loss dekh kar ruk raha tha. Isme ek khatra hota hai "Overfitting" ka—matlab model training data ko "rat" (memorize) leta hai, par naye data par fail ho jata hai.

**Early Stopping** iska smart solution hai. Yeh bilkul "Mock Test" ki tarah kaam karta hai:
1. Yeh aapke total data ka ek hissa (jaise $30\%$) chhupa leta hai jise hum Validation Data kehte hain.
2. Model bache hue $70\%$ data par seekhta (train hota) hai.
3. Har epoch ke baad, model us chhupe hue $30\%$ data par test deta hai.
4. Agar lagatar 5 baar (`n_iter_no_change`) us test (validation) mein score improve nahi hota, toh model samajh jata hai ki ab usne seekhna band kar diya hai aur sirf ratta maar raha hai. Tab wo training rok deta hai.

#### 2. Parameters Breakdown
* **`early_stopping=True`**: Yeh feature on karta hai taaki model training loss ki jagah validation score ke aadhar par training roke.
* **`validation_fraction=0.3`**: Yeh define karta hai ki total training data ka $30\%$ validation ke liye alag rakhna hai aur $70\%$ weights update karne ke liye use hoga.
* **`max_iter`, `tol`, aur `n_iter_no_change`**: Inka basic matlab wahi rehta hai, par ab yeh validation data par check kiye jate hain, na ki training data par.

#### 3. Advanced Mathematical Logic
Jab hum `early_stopping=True` set karte hain, toh internal algorithm ka mathematics is tarah badal jata hai:

**Step A: Data Splitting**
Total training set $D$ do subsets mein bat jata hai:
$$D_{fit} = (1 - validation\_fraction) \times D$$
$$D_{val} = validation\_fraction \times D$$

**Step B: Weight Optimization**
Gradient Descent apne weights ($\theta$) ko sirf $D_{fit}$ ka use karke update karta hai:
$$\theta_{t+1} = \theta_t - \eta \nabla J(\theta_t; D_{fit})$$
(Jahan $\eta$ learning rate hai aur $J$ loss function hai)

**Step C: Convergence Rule**
Har epoch $t$ ke end mein, model $D_{val}$ par error check karta hai. Model training wahin rok dega jab lagatar $n\_iter\_no\_change$ epochs tak yeh condition true ho:
$$Error_{val}(t-1) - Error_{val}(t) < tol$$

Iska matlab hai ki ab validation set par error kam hona (improve hona) lagbhag band ho gaya hai.
```python
from sklearn.linear_model import SGDRegressor
linear_regressor = SGDRegressor(
    loss="squared_error",
    early_stopping=True,
    validatiin_function=0.3,
    max_iter=100,
    tol=1e-3,
    n_iter_no_change=5
)
```

> ### Take average of all weight
``` python
from sklearn.linear model import SGDRegressor
linear_regressor = SGDRegressor (average=True)
```

```python
from sklearn.linear model import SGDRegressor
linear regressor = SGDRegressor (average=10)  # take average after 10 epoch
```

> #### how do we initlize SGD with a weight vector of previous run
* `warm_start=True`
```python
from sklearn.linear_model import SGDRegressor
linear_regressor = SGDRegressor (warm_start=True)
```

> #### access, all weights of regression , intercepts to by
```python
coeff = sgd_reg.coef_
intercept = sgd_reg.intercept_
```

In [36]:
from sklearn.linear_model import SGDRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

x , y = fetch_california_housing(as_frame=True , return_X_y=True)
x_train , x_test , y_train , y_test = train_test_split(x,y , test_size=0.2 , random_state=42)

history = []

sgd_reg = SGDRegressor(
    max_iter=1,
    tol=None,
    warm_start=True,
    penalty=None,
    learning_rate="constant",
    eta0=0.0005
)

for epoch in range(0,1000):
    sgd_reg.fit(x_train,y_train)
    y_cap = sgd_reg.predict(x_test)
    mse = mean_squared_error(y_cap,y_test)

    row = {
        "epoch":epoch,
        "mse":mse,
        "intercept":sgd_reg.intercept_[0]
    }
    
    for feature_name , weight in zip(x.columns , sgd_reg.coef_):
        row[f"weight_{feature_name}"] = weight
    history.append(row)


error_df = pd.DataFrame(history)
error_df.head(10)

,epoch,mse,intercept,weight_MedInc,weight_HouseAge,weight_AveRooms,weight_AveBedrms,weight_Population,weight_AveOccup,weight_Latitude,weight_Longitude
0,0,5.998961e+28,4.551995e+09,1.063299e+11,-1.140661e+11,-1.876824e+11,-3.072196e+10,-1.468747e+11,-7.043320e+11,-1.040189e+11,-3.012921e+11
1,1,4.275443e+30,3.086665e+09,2.395984e+11,-1.070732e+11,-2.326102e+11,-5.304624e+10,-1.149970e+12,-2.184007e+11,-1.360445e+11,-1.694165e+11
2,2,3.685648e+29,4.634405e+09,3.184723e+11,-3.099547e+11,-2.396584e+11,-6.655651e+10,3.298296e+11,3.405653e+11,-2.011909e+11,-2.392355e+11
3,3,1.525031e+30,1.613191e+09,1.411061e+11,3.436638e+11,-2.512879e+11,-4.731273e+10,-6.721226e+11,-7.012779e+11,-2.649281e+11,1.932338e+11
4,4,1.923523e+31,2.208067e+09,1.896187e+11,-6.613772e+11,-1.013507e+11,-2.581807e+10,-2.411489e+12,-1.219883e+12,-1.435734e+11,1.094784e+11
5,5,2.118535e+30,-3.666684e+08,3.599688e+11,-2.313407e+11,-7.563509e+10,-1.963589e+10,8.310961e+11,-6.658651e+11,-1.923616e+11,3.723342e+11
6,6,3.342343e+29,-8.088345e+08,3.893173e+11,1.536430e+11,1.867871e+10,-2.054526e+10,3.219711e+11,-1.832538e+12,9.193069e+10,6.050750e+10
7,7,3.333459e+29,6.541683e+09,3.868396e+11,3.505536e+11,5.837679e+09,-2.747777e+10,2.695024e+11,-8.254585e+11,3.238971e+11,-7.573530e+11
8,8,2.318855e+28,8.111728e+08,2.267164e+11,-1.277281e+11,1.058468e+11,-1.928886e+09,-9.186185e+10,-4.243304e+11,1.478749e+11,-1.321443e+11
9,9,1.529693e+30,2.177457e+09,2.028185e+11,-3.566438e+11,-3.154514e+10,-2.262594e+10,-6.882234e+11,-6.119313e+11,4.028223e+10,-1.414825e+11


In [3]:
from sklearn.linear_model import SGDRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Dataset
x, y = fetch_california_housing(
    as_frame=True,
    return_X_y=True
)

# Train Test Split
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

# Create Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SGDRegressor(random_state=42))
])

# Hyperparameter Grid
param_grid = {

    "model__loss": [
        "squared_error",
        "huber"
    ],

    "model__penalty": [
        "l1",
        "l2",
        "elasticnet"
    ],

    "model__learning_rate": [
        "constant",
        "optimal",
        "invscaling",
        "adaptive"
    ],

    "model__early_stopping": [
        True,
        False
    ]
}

# GridSearchCV
grid = GridSearchCV(

    estimator=pipe,

    param_grid=param_grid,

    cv=5,

    scoring="neg_mean_squared_error",

    n_jobs=-1,

    verbose=2
)


# Train
grid.fit(x_train, y_train)

# Best Parameters
print("Best Parameters:\n")
print(grid.best_params_)

# Best Cross Validation Score
print("\nBest CV Score:\n")
print(grid.best_score_)

# Best Model
best_model = grid.best_estimator_

# Prediction
y_pred = best_model.predict(x_test)

# Final MSE
mse = mean_squared_error(y_test, y_pred)

print("\nFinal Test MSE:\n")
print(mse)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters:

{'model__early_stopping': False, 'model__learning_rate': 'invscaling', 'model__loss': 'huber', 'model__penalty': 'l1'}

Best CV Score:

-0.5962556157268069

Final Test MSE:

0.592395389197643
